In [27]:
import datetime as dt
import pandas as pd
import numpy as np
import pymssql

from datetime import datetime, timedelta

import base64
import hashlib
from Crypto import Random
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad

In [28]:
file_ = 'VotingLuckyDraw_2026_AML.xlsx'
df = pd.read_excel(file_)
df

,BO ID,BBO ID,user name,AML No
0,68013193,4862253,Hin16,NaN
1,67954839,4814670,foxtom48,NaN
2,66544516,3784870,美富,NaN
3,65563488,2562977,chan sui wah,NaN
4,65234346,2258765,Debbie Tam 888,NaN
5,70974538,7329524,c0618,NaN
6,66264672,3164249,Hollister. L,NaN
7,59211201,885440,emilyyusm,NaN
8,354983,657458,tungli,NaN
9,66432700,3446446,hyotong,NaN


In [29]:
DB_info={'server':'192.168.61.119:7622', 'database':'openrice3', 'user':'BAReporting', 'password':'KeHeCReme8he'}
# def get_sql_data(sql,columns_name=None):
#     with pymssql.connect(DB_info['server'], DB_info['user'], DB_info['password'], 'mars') as conn:
#         cursor = conn.cursor()
#         cursor.execute(sql)
#         data = cursor.fetchall()
        
#     return pd.DataFrame(data) if columns_name== None else pd.DataFrame(data,columns=columns_name)

In [30]:
usernames = (
    df["user name"]
    .dropna()
    .astype(str)
    .str.strip()
)

usernames = usernames[usernames != ""].drop_duplicates().tolist()

if not usernames:
    view = pd.DataFrame(columns=["username", "UserId", "IMKey"])
else:
    placeholders = ", ".join(["%s"] * len(usernames))

    sql_ = f"""
    SELECT
        u.username,
        uim.UserId,
        uim.IMKey
    FROM openrice3.dbo.[User] AS u
    INNER JOIN openrice3.dbo.UserIM AS uim ON u.UserId = uim.UserId
    WHERE u.username IN ({placeholders})
    AND uim.IMTypeId in (30,31)
    """

    with pymssql.connect(DB_info["server"], DB_info["user"], DB_info["password"], "mars") as conn:
        cursor = conn.cursor()
        cursor.execute(sql_, tuple(usernames))
        data = cursor.fetchall()
        user_im = pd.DataFrame(data, columns=["username", "UserId", "IMKey"])

user_im

,username,UserId,IMKey
0,winniepingping,758999,b6+fGah/AFc5UQQrDjrO7A==
1,chankin626,59305735,OLY85cjG6dWk9sckZN4XgQ==
2,connieso138,50309804,Kw6Ifg416EWkaSBtWqP3Xw==
3,Hollister. L,66264672,B7d0dShMGlRF86M2l8Iktg==
4,lammykwan831,839592,gUsd3MHBJIgpwTJUHfmZjw==
5,catherineluff,30198083,Eztulm12NNAv1ejIXU7yig==
6,Erica Bung,65402669,tswslMyxrX7jf/yrw6OudA==
7,chan sui wah,65563488,RP1RlL8HFn21Q64AtTgfwQ==
8,calvinlai0505,482738,DUohCHpI7mGdMhM6gY9A9g==
9,ViolaCheng6729,65070708,1cXQji33jS/RPvJHhq2cwQ==


In [31]:
class AESCipher(object):
    def __init__(self, key):
        self.bs = AES.block_size
        self.key = hashlib.sha256(key.encode()).digest()    # fixed format for key & input

    def encrypt(self, raw):
        raw = self._pad(raw)
        iv = Random.new().read(AES.block_size)
        cipher = AES.new(self.key, AES.MODE_CBC, iv)
        return base64.b64encode(iv + cipher.encrypt(raw.encode()))

    def decrypt(self, enc):
        cipher = AES.new('fsQ%M4CJ^v79?g_PjX^wSg8KVE_C6vSs', AES.MODE_ECB)
        enc = base64.b64decode(enc)
        return cipher.decrypt(enc[AES.block_size:]).decode('utf-8'), cipher.decrypt(enc[AES.block_size:]), iv
        return self._unpad(cipher.decrypt(enc[AES.block_size:])).decode('utf-8')

    def _pad(self, s):
        return s + (self.bs - len(s) % self.bs) * chr(self.bs - len(s) % self.bs)

    @staticmethod
    def _unpad(s):
        return s[:-ord(s[len(s)-1:])]


def pad(s):
    bs = AES.block_size
    return s + (bs - len(s) % bs) * chr(bs - len(s) % bs)

def aml_decrypt(dori_amlid):
    a = AES.new(b'fsQ%M4CJ^v79?g_PjX^wSg8KVE_C6vSs', AES.MODE_ECB)
    s = a.decrypt(base64.b64decode(dori_amlid))
    return s[:-ord(s[len(s)-1:])].decode('utf-8')



In [32]:
df["username_lookup"] = df["user name"].astype(str).str.strip()
result = df.merge(
    user_im,
    how="left",
    left_on="username_lookup",
    right_on="username"
)
result["UserId"] = result["UserId"].astype("Int64")
has_base64_padding = result["IMKey"].astype("string").str.contains(
    "=",
    regex=False,
    na=False
)
result["AML No"] = result["IMKey"]
result.loc[has_base64_padding, "AML No"] = (
    result.loc[has_base64_padding, "IMKey"].apply(aml_decrypt)
)

result = result.drop(columns=["username_lookup", "username", "IMKey", "UserId"])
result = result.dropna(subset=["AML No"])
#result.to_excel("decrypted.xlsx", index=False)
result

,BO ID,BBO ID,user name,AML No
3,65563488,2562977,chan sui wah,1903009059
4,65234346,2258765,Debbie Tam 888,1685838547
6,66264672,3164249,Hollister. L,1744193807
7,59211201,885440,emilyyusm,1679723991
9,66432700,3446446,hyotong,1568084436
10,66579999,3460971,美味小天后,1961322244
11,66601750,3479394,or#tm66601750,1802177083
12,30198083,754339,catherineluff,1129137696
18,58407386,1047064,长长长颈鹿qin,1208893299
19,67465146,4330864,or#tm67465146,1728886736
